In [1]:
import numpy as np
import pandas as pd

# ==========================================================
# Jacobi Iterative Method
# ==========================================================
def jacobi_method(A, b, x0=None, tol=1e-8, max_iter=500):
    """
    Jacobi Iterative Method

    This function solves the linear system Ax = b using the Jacobi method.

    Idea:
    -----
    Each component of x^(k+1) is computed ONLY using values from x^(k).
    That is, all updates are done simultaneously (no immediate reuse).

    Mathematical formula:
    ---------------------
    x_i^(k+1) = (1 / a_ii) * ( b_i - sum_{j != i} a_ij * x_j^(k) )

    Parameters:
    -----------
    A : matrix (n x n)
        Coefficient matrix
    b : vector (n)
        Right-hand side
    x0 : initial guess (optional)
    tol : tolerance for stopping criterion
    max_iter : maximum number of iterations

    Returns:
    --------
    x : approximate solution
    iteration : number of iterations used
    history : list of iteration data (for tables/analysis)
    """

    # Convert inputs to numpy arrays
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    # Initial guess: if not provided, use zero vector
    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    # Extract diagonal matrix D and remainder matrix R = A - D
    D = np.diag(np.diag(A))
    R = A - D

    # Precompute inverse of diagonal (cheap operation)
    D_inv = np.diag(1 / np.diag(A))

    history = []

    # Iterative process
    for k in range(1, max_iter + 1):

        # Jacobi update (vectorized form)
        x_new = D_inv @ (b - R @ x)

        # Compute error (difference between successive iterates)
        error = np.linalg.norm(x_new - x, ord=np.inf)

        # Compute residual ||b - Ax||
        residual = np.linalg.norm(b - A @ x_new, ord=np.inf)

        # Store iteration information
        history.append({
            "Iteration": k,
            "Approximation": x_new.copy(),
            "Error (inf-norm)": error,
            "Residual (inf-norm)": residual
        })

        # Stopping criterion
        if error < tol:
            return x_new, k, history

        # Update solution
        x = x_new

    return x, max_iter, history


# ==========================================================
# Gauss-Seidel Iterative Method
# ==========================================================
def gauss_seidel_method(A, b, x0=None, tol=1e-8, max_iter=500):
    """
    Gauss-Seidel Iterative Method

    This function solves Ax = b using the Gauss-Seidel method.

    Idea:
    -----
    Unlike Jacobi, Gauss-Seidel uses newly computed values immediately.
    This typically leads to faster convergence.

    Mathematical formula:
    ---------------------
    x_i^(k+1) = (1 / a_ii) * ( b_i
                    - sum_{j<i} a_ij * x_j^(k+1)
                    - sum_{j>i} a_ij * x_j^(k) )

    Key difference:
    ---------------
    - Lower part (j < i): uses updated values
    - Upper part (j > i): uses old values

    Parameters and Returns are same as Jacobi.
    """

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float)

    history = []

    # Iteration loop
    for k in range(1, max_iter + 1):

        x_old = x.copy()

        # Update each variable sequentially
        for i in range(n):

            # Sum using already updated values
            sum1 = np.dot(A[i, :i], x[:i])

            # Sum using old values
            sum2 = np.dot(A[i, i+1:], x_old[i+1:])

            # Update x_i
            x[i] = (b[i] - sum1 - sum2) / A[i, i]

        # Compute error
        error = np.linalg.norm(x - x_old, ord=np.inf)

        # Compute residual
        residual = np.linalg.norm(b - A @ x, ord=np.inf)

        # Store iteration data
        history.append({
            "Iteration": k,
            "Approximation": x.copy(),
            "Error (inf-norm)": error,
            "Residual (inf-norm)": residual
        })

        # Stopping condition
        if error < tol:
            return x, k, history

    return x, max_iter, history


# ==========================================================
# Function to display results and comparisons
# ==========================================================
def display_results(example_name, A, b, x0=None, tol=1e-8, max_iter=500):
    """
    This function runs both methods and prints:

    - Exact solution
    - Approximate solutions
    - Number of iterations
    - Comparison table
    - First few iteration values
    """

    print("=" * 70)
    print(example_name)
    print("=" * 70)

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    # Compute exact solution using direct solver
    exact_solution = np.linalg.solve(A, b)

    # Run Jacobi
    x_jacobi, iter_jacobi, hist_jacobi = jacobi_method(A, b, x0, tol, max_iter)

    # Run Gauss-Seidel
    x_gs, iter_gs, hist_gs = gauss_seidel_method(A, b, x0, tol, max_iter)

    print("\nExact solution:")
    print(exact_solution)

    print("\nJacobi solution:")
    print(x_jacobi)
    print("Jacobi iterations:", iter_jacobi)

    print("\nGauss-Seidel solution:")
    print(x_gs)
    print("Gauss-Seidel iterations:", iter_gs)

    # Create comparison table
    comparison_table = pd.DataFrame({
        "Method": ["Jacobi", "Gauss-Seidel"],
        "Iterations": [iter_jacobi, iter_gs],
        "Final Residual": [
            hist_jacobi[-1]["Residual (inf-norm)"],
            hist_gs[-1]["Residual (inf-norm)"]
        ],
        "Approximate Solution": [x_jacobi, x_gs]
    })

    print("\nComparison Table:")
    print(comparison_table)


# ==========================================================
# Example 1 (Diagonally dominant system)
# ==========================================================
A1 = [
    [10, -1,  2,  0],
    [-1, 11, -1,  3],
    [ 2, -1, 10, -1],
    [ 0,  3, -1,  8]
]
b1 = [6, 25, -11, 15]

# ==========================================================
# Example 2 (Another stable system)
# ==========================================================
A2 = [
    [4,  1,  1],
    [2,  7,  1],
    [1, -3, 12]
]
b2 = [7, -1, 12]

# Initial guesses
x0_1 = [0, 0, 0, 0]
x0_2 = [0, 0, 0]

# Run both examples
display_results("Example 1", A1, b1, x0=x0_1)
display_results("Example 2", A2, b2, x0=x0_2)

Example 1

Exact solution:
[ 1.  2. -1.  1.]

Jacobi solution:
[ 1.  2. -1.  1.]
Jacobi iterations: 24

Gauss-Seidel solution:
[ 1.  2. -1.  1.]
Gauss-Seidel iterations: 10

Comparison Table:
         Method  Iterations  Final Residual  \
0        Jacobi          24    2.209003e-08   
1  Gauss-Seidel          10    1.388315e-09   

                                Approximate Solution  
0  [1.0000000008366594, 1.9999999985888712, -0.99...  
1  [0.9999999999868117, 1.9999999998595697, -0.99...  
Example 2

Exact solution:
[ 1.76923077 -0.74358974  0.66666667]

Jacobi solution:
[ 1.76923077 -0.74358974  0.66666667]
Jacobi iterations: 18

Gauss-Seidel solution:
[ 1.76923077 -0.74358974  0.66666667]
Gauss-Seidel iterations: 9

Comparison Table:
         Method  Iterations  Final Residual  \
0        Jacobi          18    9.542106e-09   
1  Gauss-Seidel           9    6.146728e-11   

                                Approximate Solution  
0  [1.7692307675749495, -0.7435897426394782, 0.66... 